### Testing with pytest

In [1]:
def test_addition():
    assert 2 + 2 == 4
    
def test_string_concat():
    
    result = 'hello' + ' ' + 'world'
    assert result == 'hello world'
    
print('Two tests defined')

Two tests defined


In [2]:
import ipytest

ipytest.run('-q')

..                                                                                           [100%]
2 passed in 0.04s


<ExitCode.OK: 0>

In [3]:
# def test_wrong_cal():
#     assert 2 + 3 == 10
    
# # Run the Failing Test
# ipytest.run('-q')

Testing Exceptions
* Test functions that should raise errors — using pytest.raises()

In [5]:
import pytest

def divide(a: int, b: int) -> float:
   
   if b == 0:
       raise ZeroDivisionError('Cannot divide by zero')
   return a / b


def test_divide_correct():
    assert divide(12, 3) == 4.0
    
def test_divide_by_zero():
    with pytest.raises(ZeroDivisionError):
        divide(12, 0)


# Run the Tests
ipytest.run('-q')

....                                                                                         [100%]
4 passed in 0.08s


<ExitCode.OK: 0>

**Fixtures**
* fixtures — reusable setup code shared between tests.

Basic Fixture

In [ ]:
import pytest


class FakeRetriever:
    def fetch(self, query: str) -> list[str]:
        return [f'doc about {query}']
    
class FakeLLM:
    def generate(self, prompt: str) -> str:
        return f'Answer: {prompt}'
    
    
class RAGService:
    def __init__(self, retriever, llm):
        self._retriever = retriever
        self._llm = llm
        self._is_ready = False
        
    def warm_up(self) -> None:
        self._is_ready = True
        
    def ask(self, question: str) -> str:
        if not self._is_ready:
            raise RuntimeError('Not ready')
        chunks = self._retriever.fetch(question)
        prompt = f'Context: {chunks}. Question: {question}'
        return self._llm.generate(prompt)
    
    
# --- The fixture ---
@pytest.fixture
def service():
    '''Create a warmed-up RAGService for tests'''
    
    s = RAGService(retriever=FakeRetriever(), llm=FakeLLM())
    s.warm_up()
    return s


# --- Tests use the fixture:service ---
def test_service_is_ready(service):
    '''The fixture should provide a warmed-up service.'''
    
    assert service._is_ready is True
    

def test_service_answers(service):
    '''The service should return an answer.'''
    
    answer = service.ask('How do I treat cassava?')
    assert 'Answer:' in answer
    
    
def test_service_includes_question(service):
    '''The answer should include the original question.'''
    
    answer = service.ask('What is maize smut?')
    assert 'maize smut' in answer
    
    

Run the Tests

In [7]:
ipytest.run('-q')

.......                                                                                      [100%]
7 passed in 0.06s


<ExitCode.OK: 0>

In [8]:
from pathlib import Path

# Where the file will be saved
output_path = Path(r'C:\Users\USER\rag_course\15_pure_python\agrivoice\tests\test_rag_service.py')

# The full source code — real tests for the real RAGService
source = '''"""
Tests for the RAGService class.

Uses fake dependencies to test the service in isolation.
"""

import pytest

from agrivoice.services.rag_service import RAGService


# --- Fake dependencies ---

class FakeRetriever:
    """Minimal retriever — matches RetrieverProtocol."""

    def fetch(self, query: str) -> list[str]:
        return [f'doc about {query}']


class FakeLLM:
    """Minimal LLM — matches LLMProtocol."""

    def generate(self, prompt: str) -> str:
        return f'Answer: {prompt[:40]}...'


# --- Fixture ---

@pytest.fixture
def service():
    """Warmed-up RAGService for every test that needs one."""
    s = RAGService(retriever=FakeRetriever(), llm=FakeLLM())
    s.warm_up()
    return s


# --- Tests ---

def test_service_starts_not_ready():
    """A fresh service should not be ready."""
    s = RAGService(retriever=FakeRetriever(), llm=FakeLLM())
    assert s.is_ready is False


def test_warm_up_makes_ready():
    """warm_up() should set is_ready to True."""
    s = RAGService(retriever=FakeRetriever(), llm=FakeLLM())
    s.warm_up()
    assert s.is_ready is True


def test_ask_before_warm_up_raises():
    """Calling ask() before warm_up() should raise."""
    s = RAGService(retriever=FakeRetriever(), llm=FakeLLM())
    with pytest.raises(RuntimeError):
        s.ask('anything')


def test_ask_returns_string(service):
    """A warmed-up service should return a string answer."""
    answer = service.ask('How do I treat cassava mosaic?')
    assert isinstance(answer, str)
    assert len(answer) > 0


def test_ask_includes_question(service):
    """The answer should include the original question."""
    answer = service.ask('What is maize smut?')
    assert 'maize smut' in answer
'''

# Write the file
output_path.write_text(source, encoding='utf-8')

print(f'✅ Saved: {output_path}')
print(f'   Size: {output_path.stat().st_size} bytes')

✅ Saved: C:\Users\USER\rag_course\15_pure_python\agrivoice\tests\test_rag_service.py
   Size: 1801 bytes
